In [ ]:
# -*- coding: utf-8 -*-
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or
# implied.
# See the License for the specific language governing permissions and
# limitations under the License.


In [26]:
import pandas as pd
import os
from sklearn.model_selection import train_test_split
import huggingface_hub

# Installing basic dataset

This dataset is only contain user Twitter(X) id for message and author gender.
Its purpose is to be used for the next step of dataset creation, which is to collect Twitter messages and author information using Twitter API.

on the website https://crisisnlp.qcri.org/tbcov there other options for datasets, but for my specific perpose, i choose by language and language is English.

Change iWantNewDataset to True if you want to download new dataset, otherwise it will use the existing dataset creatad by me before.
Also to create a new one you need to have Twitter(X) API bearer token and paid some money for the access to Twitter(X) API.

If you use my dataset it will save you time, money and space, because in prepared state and with only part of messages it weight only 50MB.

In [21]:
# Download original one if you want, to download original one, set iWantNewDataset to True
iWantNewDataset = False
if iWantNewDataset:
    bearer_token = None # Set your Twitter(X) API bearer token here
    if not bearer_token:
        raise ValueError("Please set your Twitter(X) API bearer token to download the dataset.")

In [ ]:
if iWantNewDataset:
    import requests
    import time
    import zipfile
    from requests.exceptions import ChunkedEncodingError, ConnectionError, ReadTimeout

    # url = "https://crisisnlp.qcri.org/data/tbcov/tbcov_country_files/united_states_of_america_tbcov.zip"
    url = "https://crisisnlp.qcri.org/data/tbcov/monthly_files/february_2020_tbcov.zip"
    zip_path = "datasets/tbcov.zip"

    os.makedirs("datasets", exist_ok=True)

    max_retries = 20
    retry_delay_sec = 3
    retries = 0

    while True:
        # Check existing file size to support resume
        existing_size = os.path.getsize(zip_path) if os.path.exists(zip_path) else 0
        headers = {"Range": f"bytes={existing_size}-"} if existing_size > 0 else {}

        try:
            with requests.get(url, stream=True, headers=headers, timeout=30) as response:
                if response.status_code == 416:
                    # Range not satisfiable — file already fully downloaded
                    print("File already fully downloaded, skipping download.")
                    break

                if response.status_code not in (200, 206):
                    raise Exception(f"Unexpected status code: {response.status_code}")

                remaining_size = int(response.headers.get("content-length", 0))
                total_size = existing_size + remaining_size if remaining_size else 0

                # If server ignores range and returns full file, restart file from scratch
                mode = "ab" if (existing_size > 0 and response.status_code == 206) else "wb"
                downloaded = existing_size if mode == "ab" else 0

                if mode == "ab" and existing_size > 0:
                    print(f"Resuming download from {existing_size / (1024*1024):.2f} MB...")

                with open(zip_path, mode) as file:
                    for chunk in response.iter_content(chunk_size=1024 * 1024):  # 1MB chunks
                        if chunk:
                            file.write(chunk)
                            downloaded += len(chunk)
                            if total_size > 0:
                                print(
                                    f"\rDownloaded: {downloaded / (1024*1024):.2f} MB / {total_size / (1024*1024):.2f} MB",
                                    end="",
                                    flush=True,
                                )
                            else:
                                print(
                                    f"\rDownloaded: {downloaded / (1024*1024):.2f} MB",
                                    end="",
                                    flush=True,
                                )

                # Stream finished successfully for this request
                retries = 0
                print("\nDownload complete!")
                break

        except (ChunkedEncodingError, ConnectionError, ReadTimeout) as e:
            retries += 1
            if retries > max_retries:
                raise Exception(f"Download failed after {max_retries} retries: {e}") from e

            current_size = os.path.getsize(zip_path) if os.path.exists(zip_path) else 0
            print(
                f"\nConnection interrupted ({type(e).__name__}). "
                f"Retry {retries}/{max_retries} from {current_size / (1024*1024):.2f} MB..."
            )
            time.sleep(retry_delay_sec)

    # Unzip the downloaded file
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall("datasets/")
    print("Unzipping complete!")

    # Remove the zip file after extraction
    os.remove(zip_path)
    print("Zip file removed!")

    # Preprocessing the dataset to form where we can merge it with text for tweets
    import xdk
    dir = "./datasets/"
    tmp_file = os.path.join(dir, "tmp.csv")
    final_file = os.path.join(dir, "final.csv")
    header = True
    for file in os.listdir(dir):
        # Process each TSV file
        if not file.endswith(".tsv"):
            continue
        
        for rows in pd.read_csv(os.path.join(dir, file), sep="\t", chunksize=10000):
            rows[["tweet_id", "label"]] \
                .rename(columns={"label": "gender"}) \
                .to_csv(tmp_file, index=False, mode="a", header=header)
            header = False

    # Read text from twitter(X) API and merge it with the labels
    header = True
    for rows in pd.read_csv(tmp_file, chunksize=10000):
        bearer_token = "YOUR_BEARER"
        client = xdk.Client(bearer_token=bearer_token)
        response = client.posts.get_by_ids(ids=rows["tweet_id"].tolist(), tweet_fields=["text"])
        if response.data is None:
            continue
        rows["text"] = rows["tweet_id"].map({post.id: post.text for post in response.data})
        
        # Save the updated rows back to the temporary file
        rows.to_csv(tmp_file, index=False, mode="a", header=header)
        header = False

    # Remove the temporary file after processing
    os.remove(tmp_file)
    for file in os.listdir(dir):
        if file.endswith(".tsv"):
            os.remove(os.path.join(dir, file))

    df = pd.read_csv(final_file)
    df.drop_duplicates(inplace=True)
    df.dropna(inplace=True)
    df["text_length"] = df["text"].apply(len)
    df.sort_values(by="text_length", ascending=False, inplace=True)
    df.drop_duplicates(subset=["text"], inplace=True, keep="first")
    df.drop(columns=["text_length"], inplace=True)
    df.to_csv(os.path.join(dir, "one_tweet_dataset_full.csv"), index=True)
    os.remove(final_file)
    splits = {'full': 'one_tweet_dataset_full.csv','train': 'one_tweet_dataset_train.csv', 'validation': 'one_tweet_dataset_val.csv', 'test': 'one_tweet_dataset_test.csv'}
    train_df, test_df = train_test_split(df, test_size=0.1)
    train_df, val_df = train_test_split(train_df, test_size=0.1)
    train_df.to_csv(os.path.join(dir, splits['train']), index=False)
    val_df.to_csv(os.path.join(dir, splits['validation']), index=False)
    test_df.to_csv(os.path.join(dir, splits['test']), index=False)
    print("dataset with one tweet per author is ready!")


if iwantNewDataset is False, it will use the existing dataset, that i created for initial stage of this project.

In [32]:
if not iWantNewDataset:
    from huggingface_hub import hf_hub_download
    
    splits = {'full': 'one_tweet_dataset_full.csv','train': 'one_tweet_dataset_train.csv', 'validation': 'one_tweet_dataset_val.csv', 'test': 'one_tweet_dataset_test.csv'}
    
    # Download the dataset file from Hugging Face Hub
    for name in splits.values():
        file_path = hf_hub_download(
            repo_id="qg2020252627/twitter_author_profiling_by_gender_nlp",
            filename=name,
            repo_type="dataset"
        )
        df = pd.read_csv(file_path)
        df.to_csv(os.path.join("datasets", name), index=False)